In [153]:
import pandas as pd
import numpy as np
import ast
from sklearn.metrics.pairwise import cosine_similarity

This notebook is to solve a problem that we have in 01-mca-soc-crosswalk. Please refer to that notebook first. This only has the number of 00 because this is to analyze previous work where we mapped the MCA titles to SOC codes by just taking the last code.

In [95]:
# Open up the data
filepath = '../data/wadhwani/FINAL (WIP)-MCA Job list August 2025.xlsx'
final_jobs_df = pd.read_excel(filepath)
sector_reports_jobs_df = pd.read_excel(filepath, sheet_name='Pass 3 - Sector reports', dtype={'PSOC Code': str})

# Clean up the job titles
final_jobs_df['Job Title'] = final_jobs_df['Job Title'].str.strip()
sector_reports_jobs_df['Job Title'] = sector_reports_jobs_df['Job Title'].str.strip()

# 1. Mapping MCA titles to PSOC to 2010 O*NET SOC

I am asking the folllowing questions:
1. How many ISCO codes have multiple SOC mappings?
2. Among those, how often is the last SOC "All Other"?

In [96]:
# define the psoc mapping already present
psoc_mapping = dict(zip(sector_reports_jobs_df['Job Title'], sector_reports_jobs_df['PSOC Code']))

# Add in the final jobs the PSOC
final_jobs_df['PSOC Code'] = final_jobs_df['Job Title'].map(psoc_mapping)

# Get the data
filename = '../data/labor_codes/2022-Updates-to-the-2012-PSOC.xlsx'
relevant_cols = [2, 5]
names = ['PSOC', 'ISCO']
df_maps = pd.read_excel(
    filename, 
    usecols=relevant_cols, 
    names=names,
    sheet_name=None,
    dtype={"PSOC": str, "ISCO": str}
    )

# for each df_map in df_maps, get their jobs to ISCO pairs
psoc_isco = {}
for _, df_map in df_maps.items():
    df_map.dropna(inplace=True)
    mapping = dict(zip(df_map['PSOC'], df_map['ISCO']))
    psoc_isco.update(mapping)

# Correct the wrong PSOC codes
corrected_psoc_codes = {
    '4111' : '4110',
    '3144' : '2132',
    '7414' : '7412',
    '6110' : '6121'
}
final_jobs_df['PSOC Code'] = final_jobs_df['PSOC Code'].replace(corrected_psoc_codes)

# Convert from PSOC to ISCO
final_jobs_df['ISCO Code'] = final_jobs_df['PSOC Code'].map(psoc_isco)
# Clean up the ISCO codes by just only keeping the digits
final_jobs_df['ISCO Code'] = final_jobs_df['ISCO Code'].str.extract(r'(\d+)', expand=False)

# BUILD THE MACHINERY FOR THE ISCO TO 2010 SOC crosswalk
filename = '../data/labor_codes/ISCO_SOC_Crosswalk.xls'
isco_soc_df = pd.read_excel(filename,
                            usecols=[0, 3, 4],
                            names=['ISCO', 'SOC', 'SOC_Title'],
                            skiprows=6,
                            dtype={'ISCO': str, 'SOC': str, 'SOC_Title': str})
isco_soc_df['SOC'] = isco_soc_df['SOC'].str.strip()
isco_soc_df['ISCO'] = isco_soc_df['ISCO'].str.strip()

# create the mappings for isco -> 2008 soc code 
# THIS IS WHERE IT IS UNSOPHISTICATED, i legit just took the last one
isco_soc = dict(zip(isco_soc_df.ISCO, isco_soc_df.SOC))
soc_code_title = dict(zip(isco_soc_df.SOC, isco_soc_df.SOC_Title))

# fix the error
final_jobs_df['ISCO Code'] = final_jobs_df['ISCO Code'].replace({'612' : '6121'})

In [97]:
isco_soc_df['is_all_other'] = (
    isco_soc_df['SOC_Title']
    .str.contains('All Other', case=False, na=False)
)

# identify ISCO codes with multiple SOC mappings
multi_isco = (
    isco_soc_df
    .groupby('ISCO')
    .filter(lambda x: len(x) > 1)
)

# reproduce exactly what the mapping of choosing the last code
last_mapping = (
    isco_soc_df
    .groupby('ISCO', sort=False)
    .tail(1)
).copy()

last_mapping['is_all_other'] = (
    last_mapping['SOC_Title']
    .str.contains('All Other', case=False, na=False)
)

In [98]:
num_isco_codes = len(set(isco_soc_df.ISCO))
num_multi_isco_codes = len(set(isco_soc_df.ISCO))

print(f"There are {num_isco_codes} ISCO codes where "
      f"{num_multi_isco_codes} of them can map to several distinct SOC codes.")

There are 438 ISCO codes where 438 of them can map to several distinct SOC codes.


In [99]:
# i am looking at specifically ISCO codes that can map to multiple SOC codes
multi_last_mapping = (
    isco_soc_df[isco_soc_df['ISCO'].isin(multi_isco['ISCO'])]
    .groupby('ISCO', sort=False)
    .tail(1)
)

pct_last_is_all_other = (
    multi_last_mapping['is_all_other'].mean()
)

print(
    f"If we determine the ISCO-to-SOC mapping by choosing the last code listed, "
    f"then {pct_last_is_all_other * 100:.0f}% of cases with multiple possible "
    f"SOC mappings result in a general ('All Other') occupation."
)

print(
    f"The remaining {(1 - pct_last_is_all_other) * 100:.0f}% of cases with "
    f"multiple possible SOC mappings result in a non-general occupation."
)

If we determine the ISCO-to-SOC mapping by choosing the last code listed, then 20% of cases with multiple possible SOC mappings result in a general ('All Other') occupation.
The remaining 80% of cases with multiple possible SOC mappings result in a non-general occupation.


Below are the ISCO codes that have multiple possible SOC mappings to just show examples.

In [100]:
multi_last_mapping.head(20)

,ISCO,SOC,SOC_Title,is_all_other
7,0110,55-1019,Military Officer Special and Tactical Operatio...,True
10,0210,55-2013,First-Line Supervisors of All Other Tactical O...,True
19,0310,55-3019,Military Enlisted Tactical Operations and Air/...,True
23,1112,11-9161,Emergency Management Directors,False
25,1113,11-1031,Legislators,False
28,1114,11-9199,"Managers, All Other",True
30,1120,11-1021,General and Operations Managers,False
34,1212,11-3131,Training and Development Managers,False
41,1219,11-9199,"Managers, All Other",True
43,1221,11-2022,Sales Managers,False


# 2. Mapping 2010 SOC to 2019 SOC

This is where the previous work stopped because I only mapped to the 2010 SOC. I will be continuing the work and map until the 2019 SOC.

In [111]:
# 2010 to 2018
filename = '../data/labor_codes/soc_2010_to_2018_crosswalk.xlsx'
soc_2010_2018_df = pd.read_excel(
    filename,
    usecols=[0, 2],
    names =[2010, 2018],
    dtype={2010:'str', 2018:'str'},
    skiprows=9
)
soc_2010_2018_df[2010] = soc_2010_2018_df[2010].str.strip()
soc_2010_2018_df[2018] = soc_2010_2018_df[2018].str.strip()
soc_2010_2018_map = dict(zip(soc_2010_2018_df[2010], soc_2010_2018_df[2018]))

# 2018 to 2019
crosswalk_2018_2019_df = pd.read_csv('../data/labor_codes/2019_to_SOC_Crosswalk.csv')
soc_2018_2019_map = dict(zip(crosswalk_2018_2019_df['2018 SOC Code'], crosswalk_2018_2019_df['O*NET-SOC 2019 Code']))


# 2010 to 2019
soc_2010_2019_map = {soc_2010 : soc_2018_2019_map[soc_2018] for soc_2010, soc_2018 in soc_2010_2018_map.items()}

# isco to onet 2019
isco_soc_2019_map = {isco : soc_2010_2019_map[soc_2010] for isco, soc_2010 in isco_soc.items()}

In [125]:
final_jobs_df['SOC Code'] = final_jobs_df['ISCO Code'].map(isco_soc_2019_map)

# 2. Choose the best possible title given the code

In [133]:
def create_mapping(
    df_filepath: str,
    group_column: str,
    value_column: str,
    is_numeric=True
) -> dict[str, np.ndarray]:
    """
    Create a mapping from each group value to its corresponding row values.

    Each unique value in `group_column` becomes a key in the returned
    dictionary. The corresponding value is a NumPy array containing the
    values from `value_column` for rows belonging to that group.

    Args:
        df_filepath: Path to the CSV file containing the data.
        group_column: Column used to group the rows.
        value_column: Column containing the values to collect.

    Returns:
        A dictionary mapping each group value to a NumPy array of
        corresponding values from `value_column`.
    """
    # open the data
    df = pd.read_csv(df_filepath)
    if is_numeric:
        df[value_column] = df[value_column].apply(ast.literal_eval)

    # create the mapping
    if is_numeric:
        return {
            str(group_value): np.array(group_values.tolist(), dtype=np.float32)
            for group_value, group_values in df.groupby(group_column)[value_column]
        }
    else:
        return {
            str(group_value): group_values.tolist()
            for group_value, group_values in df.groupby(group_column)[value_column]
        }

In [ ]:
# Create mappings for MCA and SOC title embeddings
embedded_mca_title_map = create_mapping(
    "../data/auxiliary/embedded_mca_title.csv",
    "Job Title",
    "Job Title Embedded",
)
embedded_soc_title_map = create_mapping(
    "../data/auxiliary/embedded_soc_title.csv",
    "O*NET-SOC Code",
    "Job Title Embedded",
)

code_titles_map = create_mapping(
    "../data/auxiliary/embedded_soc_title.csv",
    "O*NET-SOC Code",
    "Job Title",
    is_numeric=False
)

In [148]:
final_jobs_df["Job Title Embedded"] = final_jobs_df["Job Title"].apply(
    lambda title: embedded_mca_title_map[title]
)

final_jobs_df["SOC Titles Embedded"] = final_jobs_df["SOC Code"].apply(
    lambda soc_code: embedded_soc_title_map[soc_code]
)

In [175]:
def estimate_representative_soc_codes(job, verbose=False):
    """
    Rank candidate SOC codes from most to least representative.

    Each candidate SOC code is evaluated using:
    1. Task similarity: mean of the best cosine similarity for each
       PSOC task against the candidate SOC's tasks.
    2. Title similarity: maximum cosine similarity between the PSOC
       job title and any of the candidate SOC's titles.

    The final score is a weighted combination of task and title similarity.

    Returns:
        Tuple containing:
        - List of SOC codes sorted by score
        - List of best-matching SOC titles corresponding to each SOC code
    """

    # If there is only one 2019 SOC code, leave it unchanged
    embedded_job_title = np.array(
        job['Job Title Embedded']
    ).reshape(1, -1)

    soc_scores = []

    soc_code = job['SOC Code']
    #print(soc_code)


    # Evaluate every candidate SOC code
    for embedded_soc_titles in job['SOC Titles Embedded']:
        # -------------------------
        # Title similarity
        # -------------------------
        embedded_soc_titles = np.array(
            embedded_soc_titles
        )

        if embedded_soc_titles.ndim == 1:
            embedded_soc_titles = embedded_soc_titles.reshape(1, -1)

        title_similarity_matrix = cosine_similarity(
            embedded_job_title,
            embedded_soc_titles
        )

        best_title_index = title_similarity_matrix.argmax()

        title_similarity = title_similarity_matrix[
            0,
            best_title_index
        ]

        # Get actual title using the SOC code
        best_soc_title = code_titles_map[soc_code][best_title_index]

        # -------------------------
        # Final score
        # -------------------------

        score = title_similarity

        soc_scores.append({
            'Best SOC Title': best_soc_title,
            'Title Similarity': title_similarity,
            'Score': score
        })

    # Sort from most likely to least likely
    soc_scores = sorted(
        soc_scores,
        key=lambda x: x['Score'],
        reverse=True
    )


    result_soc_titles = [
        soc_score['Best SOC Title']
        for soc_score in soc_scores
    ]
    
    return result_soc_titles

In [179]:
# get the best representative soc code
sorted_soc_titles = final_jobs_df.apply(
    estimate_representative_soc_codes,
    axis=1,
)
final_jobs_df['SOC Title'] = sorted_soc_titles.apply(
    lambda soc_titles : soc_titles[0]
)

In [ ]:
final_jobs_df.to_csv('../data/')

,Job Title,Educational Qualification,Job Sector,Educational Pathway,HEI with PRC Exam,Some HEI,Job Subsector,PSOC Code,ISCO Code,SOC Code,Job Title Embedded,SOC Titles Embedded,SOC Title
0,Operations Assistant,Bachelor in Business Engineering,Administrative and Support Service Activities,Higher Education,No,Yes,Office administrative and support activities,3343,3343,43-6011.00,"[[0.0016874041, 0.0040603755, -0.027565202, -0...","[[-0.010671205, -0.0007899429, -0.024253516, 0...",Administrative Aide
1,Junior Business News Writer,Bachelor in Business Journalism,Information and Communication,Higher Education,No,No,"Publishing of books, periodicals and other pub...",2642,2642,27-3041.00,"[[0.050976705, 0.0026855702, 0.024874574, -0.0...","[[0.08567805, 0.05620133, -0.018109154, -0.040...",Acquisitions Editor
2,Logistics Assistant,Bachelor in Port Administration,Transportation and Storage,Higher Education,No,Yes,Support activities for transportation,4321,4321,43-5111.00,"[[0.030527376, 0.009583268, -0.028841428, 0.00...","[[-0.027200522, 0.059560183, -0.007602882, 0.0...",Aircraft Shipping Checker
3,Financial Auditor,Bachelor of Accountancy,Financial and Insurance Activities,Higher Education,Yes,No,"Accounting, bookkeeping and auditing activitie...",2411,2411,13-2082.00,"[[0.009616867, 0.060093705, -0.019835798, 0.01...","[[-0.019520573, -0.015197942, 0.00597167, -0.0...",Corporate Tax Preparer
4,Tax Accountant,Bachelor of Accounting Technology,Financial and Insurance Activities,Higher Education,Yes,No,"Accounting, bookkeeping and auditing activitie...",2411,2411,13-2082.00,"[[0.02102572, 0.047098275, -0.017406572, 0.012...","[[-0.019520573, -0.015197942, 0.00597167, -0.0...",Corporate Tax Preparer
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1146,Backend Developer,Web Development (Back-End) NC III,Information and Communication,Not Higher Education,No,No,"Computer programming, consultancy and related ...",2513,2513,15-1255.01,"[[0.03703596, 0.014578998, -0.053859293, -0.00...","[[0.036826525, 0.01760701, -0.07795979, 0.0226...",Computer Game Designer
1147,Front-End Web Developer,Web Development (Front-End) NC III,Information and Communication,Not Higher Education,No,No,"Computer programming, consultancy and related ...",2513,2513,15-1255.01,"[[0.056063127, -0.040090036, -0.04730732, -0.0...","[[0.036826525, 0.01760701, -0.07795979, 0.0226...",Computer Game Designer
1148,Wind Turbine Technician,Wind Turbine Maintenance Services Level III,"Electricity, Gas, Steam and Air Conditioning S...",Not Higher Education,No,No,"Electric power generation, transmission and di...",3113,3113,17-3024.01,"[[-0.05004683, -0.032454126, -0.039435856, 0.0...","[[-0.032227904, -0.010201533, -0.028677158, -0...",Assembly Technician
1149,Wood Carver,Wood Carving NC II,Manufacturing,Not Higher Education,No,No,Manufacture of furniture,7317,7317,51-7099.00,"[[0.0020079948, 0.05244599, 0.009716483, 0.018...","[[-0.056944046, -0.011701389, -0.0015453292, 0...",Accordion Maker
